# Análisis Exploratorio de Datos (EDA) — Análisis de Estacionalidad de Pernoctaciones
**Analista: Rubén Serra Sanz | 11 Mayo 2026**

---

## Índice
1. Objetivos
2. Extracción y carga de datos de fuentes externas (INE)
3. Descripción del dataset
4. Información general del dataset
5. Metodología de Análisis


## 1. Objetivos del Análisis


• Identificar los patrones de demanda (pernoctaciones) en Madrid, Cataluña, Valencia, Baleares y Andalucía.

• Determinar los meses de máxima ocupación (picos) y mínima (valles) para cada región.

• Proporcionar evidencia basada en datos para el calendario de ofertas de la empresa.

In [25]:
df_pernoctaciones

,Totales Territoriales,Comunidades y Ciudades Autónomas,Provincias,Viajeros y pernoctaciones,Residencia: Nivel 1,Residencia: Nivel 2,Periodo,Total
0,Total Nacional,01 Andalucía,NaN,Pernoctaciones,Total,NaN,2026M03,4.036.015
1,Total Nacional,01 Andalucía,NaN,Pernoctaciones,Total,NaN,2026M02,2.833.842
2,Total Nacional,01 Andalucía,NaN,Pernoctaciones,Total,NaN,2026M01,2.485.158
3,Total Nacional,01 Andalucía,NaN,Pernoctaciones,Total,NaN,2025M12,2.693.046
4,Total Nacional,01 Andalucía,NaN,Pernoctaciones,Total,NaN,2025M11,3.145.703
...,...,...,...,...,...,...,...,...
250,Total Nacional,"13 Madrid, Comunidad de",NaN,Pernoctaciones,Total,NaN,2022M05,2.052.491
251,Total Nacional,"13 Madrid, Comunidad de",NaN,Pernoctaciones,Total,NaN,2022M04,2.004.575
252,Total Nacional,"13 Madrid, Comunidad de",NaN,Pernoctaciones,Total,NaN,2022M03,1.720.481
253,Total Nacional,"13 Madrid, Comunidad de",NaN,Pernoctaciones,Total,NaN,2022M02,1.435.535


## 2. Extracción y carga de datos de fuentes externas (INE)

Conexión a la base de datos de una fuente secundaria (INE)

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import getpass
import sweetviz as sv
df_pernoctaciones = pd.read_csv("C:\Simulador_Análisis_datos\Sprint 4\Pernoctaciones_CCAA.csv", encoding='latin-1', sep=';')

<>:7: SyntaxWarning: invalid escape sequence '\S'
<>:7: SyntaxWarning: invalid escape sequence '\S'
C:\Users\ruben\AppData\Local\Temp\ipykernel_12328\1147148183.py:7: SyntaxWarning: invalid escape sequence '\S'
  df_pernoctaciones = pd.read_csv("C:\Simulador_Análisis_datos\Sprint 4\Pernoctaciones_CCAA.csv", encoding='latin-1', sep=';')


## 3. Descripción del dataset

El dataset procede del Instituto Nacional de Estadística (INE), concretamente de la Encuesta de Ocupación Hotelera (EOH). Se han extraído las series mensuales de pernoctaciones para las comunidades autónomas donde StaySpain tiene presencia activa.

Descripción de las variables extraídas

- Totales Territoriales: Columna de agregación de nivel superior que contiene los datos consolidados a nivel nacional.

- Provincias: Desglose geográfico de segundo nivel (no se utilizará para este análisis, ya que el objetivo son las CCAA).

- Viajeros y pernoctaciones: Variable de concepto que define qué métrica se está midiendo en esa fila.

- Residencia (Nivel 1 y Nivel 2): Variables de segmentación de procedencia (Nacional/Extranjero). Aunque el foco es el mes de visita, estas columnas son vitales para entender el perfil del viajero.

- Periodo: Variable temporal clave con formato YYYY'M'MM (ej: 2023M12).

- Total: Columna numérica que contiene el valor absoluto de la medición (el número de pernoctaciones o viajeros).

> ⚠️ **Limpieza de Prefijos:** La columna Comunidades y Ciudades Autónomas requiere la eliminación de los códigos numéricos para que las visualizaciones de negocio sean legibles.

> ⚠️ **Parseo de Fecha:** La columna Periodo debe ser transformada de string a objeto datetime eliminando la letra "M" del formato original del INE para permitir el análisis de estacionalidad mensual.

> ⚠️ **Limpieza de Formato Numérico (Separador de Miles):** La columna Total presenta los valores con un punto (.) como separador de miles (ej: 1.250.000). Este carácter provoca que Pandas interprete la columna como tipo object (texto) o maneje erróneamente los decimales. Se requiere eliminar los puntos de forma masiva y convertir la columna a tipo entero (int64) antes de cualquier cálculo estadístico.

## 4. Información general del dataset

In [24]:
# Dimensiones y tipos de datos
print(f"Filas: {df_pernoctaciones.shape[0]} | Columnas: {df_pernoctaciones.shape[1]}")
print()
print(df_pernoctaciones.info())

Filas: 255 | Columnas: 8

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255 entries, 0 to 254
Data columns (total 8 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Totales Territoriales             255 non-null    object 
 1   Comunidades y Ciudades Autónomas  255 non-null    object 
 2   Provincias                        0 non-null      float64
 3   Viajeros y pernoctaciones         255 non-null    object 
 4   Residencia: Nivel 1               255 non-null    object 
 5   Residencia: Nivel 2               0 non-null      float64
 6   Periodo                           255 non-null    object 
 7   Total                             255 non-null    object 
dtypes: float64(2), object(6)
memory usage: 16.1+ KB
None


Los tipos de datos parecen correctos en general. Sin embargo hay variables que requerirán transformación antes del análisis:

- Totales Territoriales. Representa el nivel de agregación nacional. Se rellena de forma automática al seleccionar los datos que quieres descargar del INE. Esta columna no nos sirve para nuestro análisis
- Comunidades y Ciudades Autónomas: Dato cualitativo nominal. Es la dimensión geográfica principal del análisis. Contiene texto mixto (código + nombre).
- Provincias: Dato cualitativo nominal. Actualmente sin información relevante (columna de nulos).
- Viajeros y pernoctaciones: Dato cualitativo nominal. Actúa como filtro para definir la métrica de negocio. El nombre de la columna incluye viajeros y pernoctaciones, pero realmente esta columna solamente tiene información de pernoctaciones.
- Residencia: Nivel 1 y Nivel 2: Datos cualitativos nominales. Definen el perfil de procedencia del viajero. En nuestro análisis se obviará esta información.
- Periodo: Actualmente cargada como Object (String). Su contenido sigue el patrón YYYY'M'MM. Técnicamente es una serie temporal discreta que debe ser convertida a datetime64 para permitir cálculos de estacionalidad.
- Total: Actualmente interpretada como Object (String) debido a la presencia de puntos (.) como separadores de miles. Representa una variable cuantitativa discreta (conteo de pernoctaciones). Su estado actual impide realizar operaciones matemáticas (sumas, medias, etc.).



> ⚠️ **Eliminación de Columnas Irrelevantes:** Las columnas Provincias y Residencia: Nivel 2 deben ser descartadas del dataframe final, ya que solo contienen valores nulos y aumentarían el ruido en el análisis.

> ⚠️ **Limpieza de Prefijos:** La columna Comunidades y Ciudades Autónomas requiere la eliminación de los códigos numéricos para que las visualizaciones de negocio sean legibles.

> ⚠️ **Parseo de Fecha:** La columna Periodo debe ser transformada de string a objeto datetime eliminando la letra "M" del formato original del INE para permitir el análisis de estacionalidad mensual.

> ⚠️ **Limpieza de Formato Numérico (Separador de Miles):** La columna Total presenta los valores con un punto (.) como separador de miles (ej: 1.250.000). Este carácter provoca que Pandas interprete la columna como tipo object (texto) o maneje erróneamente los decimales. Se requiere eliminar los puntos de forma masiva y convertir la columna a tipo entero (int64) antes de cualquier cálculo estadístico.

## 5. Metodología de Análisis

Para dar respuesta a la pregunta de negocio sobre la necesidad de ajustar las ofertas según los meses de visita, se seguirá un proceso estructurado de tratamiento y análisis de datos. Dado que el dataset original presenta una estructura jerárquica y formatos no estandarizados, la metodología se divide en las siguientes fases:

## A. Filtrado Estratégico y Segmentación

    • Selección de CCAA: Se filtrará el dataset para trabajar exclusivamente con las 5 comunidades autónomas donde StaySpain tiene presencia: Andalucía, Baleares, Cataluña, C. Valenciana y Madrid.

    • Limpieza de Jerarquías: Se eliminarán los registros correspondientes a "Totales Nacionales" y "Provincias" para evitar la duplicidad de datos en el análisis mensual.

    • Aislamiento de Métricas: Se seleccionarán únicamente los registros de "Pernoctaciones", descartando los datos de viajeros para centrar el estudio en la ocupación real.

## B. Ingeniería de Características (Feature Engineering)

    • Normalización Temporal: Conversión del campo Periodo (formato string) a objeto datetime. Esto permitirá realizar agrupaciones por mes y año, identificando ciclos estacionales.

    • Sanitización Numérica: Limpieza de la columna Total mediante la eliminación de separadores de miles para su conversión a tipo entero (int64).

## C. Análisis de Estacionalidad y Demanda
    • Identificación de Picos y Valles: Se analizará la fluctuación mensual de pernoctaciones para detectar los meses de máxima y mínima demanda en cada región.

    • Análisis Comparativo: Se comparará el "latido" turístico entre comunidades (ej. la estacionalidad de sol y playa frente a la urbana de Madrid).

## D. Visualización y Conclusiones de Negocio
    • Utilización de gráficos de líneas para observar tendencias temporales y heatmaps para identificar visualmente las ventanas de oportunidad para el lanzamiento de ofertas y promociones específicas.